In [1]:
!pip install "langchain==0.3.27" "langchain-community==0.3.27" "langchain-text-splitters==0.3.11" "langchain-experimental==0.3.4" "langchain-core==0.3.78" sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.1
    Uninstalling packaging-26.1:
      Successfully uninstalled packaging-26.1
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.15
    Uninstalling lan

In [1]:
# full_pipeline.py
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import CrossEncoder


In [2]:
# Config
EMBEDDING_MODEL    = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL     = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RETRIEVAL_K        = 20          # retrieve wide
RERANK_TOP_N       = 5           # pass narrow
CONFIDENCE_THRESH  = 0.70
COMPRESSION_THRESH = 0.76

In [3]:
# Step 1: Load data
df = pd.read_csv("/content/customer_support_tickets.csv")
documents = df["Ticket Description"].dropna().tolist()
print(f"Loaded {len(documents)} tickets")


Loaded 8469 tickets


In [4]:
# Step 2: Semantic chunking
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
splitter   = SemanticChunker(embeddings=embeddings, breakpoint_threshold_type="percentile")

chunks = []
for doc in documents[:500]:
    chunks.extend(splitter.split_text(doc))
print(f"Semantic chunks created: {len(chunks)}")


/tmp/ipykernel_8367/3537206726.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.war

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic chunks created: 1000


In [5]:
# Step 3: Index
vectorstore    = FAISS.from_texts(chunks, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
reranker       = CrossEncoder(RERANKER_MODEL)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [6]:
# Step 4: Compression retriever
compressor = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=COMPRESSION_THRESH
)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)


In [16]:
# Step 5: Production query function
def production_rag(query: str) -> dict:

    # 5a. Retrieve (wide)
    candidates = vectorstore.similarity_search_with_score(query, k=RETRIEVAL_K)

    # 5b. Confidence gate
    confident = [(doc, s) for doc, s in candidates if s >= CONFIDENCE_THRESH]
    if not confident:
        return {
            "answer"    : "I don't have reliable information on this. Please contact support@company.com.",
            "source"    : "fallback",
            "best_score": round(float(candidates[0][1]), 4) if candidates else None
        }

    # 5c. Rerank (narrow)
    docs    = [doc for doc, _ in confident]
    pairs   = [(query, doc.page_content) for doc in docs]
    scores  = reranker.predict(pairs)
    reranked = [
    doc for _, doc in sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True,
    )
][:RERANK_TOP_N]


    # 5d. Build context
    context = "\n\n---\n\n".join([doc.page_content for doc in reranked])

    # 5e. Generate (replace with your LLM call)
    return {
        "answer"     : f"[LLM would generate here using context below]\n\n{context[:500]}...",
        "source"     : "rag",
        "chunks_used": len(reranked),
        "best_score" : round(float(confident[0][1]), 4)
    }


In [17]:
# Test
test_queries = [
    "What is the refund policy for billing errors?",
    "How do I cancel my subscription?",
    "What is the capital of Mars?",   # out-of-domain → should fallback
]

for q in test_queries:
    result = production_rag(q)
    print(f"\nQuery  : {q}")
    print(f"Source : {result['source'].upper()}")
    print(f"Answer : {result['answer'][:200]}...")
    print("-" * 70)


Query  : What is the refund policy for billing errors?
Source : FALLBACK
Answer : I don't have reliable information on this. Please contact support@company.com....
----------------------------------------------------------------------

Query  : How do I cancel my subscription?
Source : RAG
Answer : [LLM would generate here using context below]

It's the same as having an issue with buying it as I only need to fill out the address on the bill. I didn't realize my order was delayed. I'm unable to ...
----------------------------------------------------------------------

Query  : What is the capital of Mars?
Source : RAG
Answer : [LLM would generate here using context below]

What does it mean?

---

What does it mean?

---

What does it mean?

---

What does it mean?

---

What does it mean?......
----------------------------------------------------------------------
